# 🧠 Suicide & Depression Post Detection

**Dataset**: [Suicide Watch](https://www.kaggle.com/datasets/nikhileswarkomati/suicide-watch)
— 232,074 Reddit posts labelled `suicide` or `non-suicide`


In [ ]:
# ── ── 0. Setup & Installation ────────────────────────────────────────────────
# Cell 0a — Install dependencies
!pip install -q groq
# !pip install -q -r requirements.txt

# Cell 0b — Mount Drive (optional, for persistent storage in Colab)
# from google.colab import drive
# drive.mount('/content/drive')

import os, sys, warnings
warnings.filterwarnings('ignore')
# Add the project root to the system path to allow importing from 'src'
sys.path.insert(0, '.') # Reverting to adding the current directory

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

# Download NLTK resources required by src modules *before* importing them
import nltk
nltk.download('stopwords')

# Project modules
from src.preprocessing import download_nltk_resources, preprocess_dataframe, preprocess_text
from src.features import (
    build_tfidf_vectorizer, build_feature_matrix,
    extract_handcrafted_features, HANDCRAFTED_FEATURE_NAMES,
    save_vectorizer,
)
from src.train_ml import split_data, train_all_models, save_models
from src.evaluate import full_evaluation, compare_models
from src.llm import explain_prediction

os.makedirs('outputs/plots', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

download_nltk_resources() # Keep this in case other NLTK resources are downloaded here
print("✓ Setup complete")

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d nikhileswarkomati/suicide-watch -p data/ --unzip


## 1. Data Loading
Download the Suicide Watch dataset from Kaggle using the Kaggle API.

In [ ]:
# ── ── 1. Data Loading ────────────────────────────────────────────────────────
# ── Option B: Manual upload ───────────────────────────────────────────────────
# Upload the CSV manually and place it at: data/Suicide_Detection.csv

DATA_FILE = 'data/Suicide_Detection.csv'


df_raw = pd.read_csv(DATA_FILE)
print(f"Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(df_raw.head(3))

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── ── 2a. Basic Dataset Info ──────────────────────────────────────────────────
print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"\nShape          : {df_raw.shape}")
print(f"Columns        : {list(df_raw.columns)}")
print(f"\nData types:\n{df_raw.dtypes}")
print(f"\nMissing values:\n{df_raw.isnull().sum()}")
print(f"\nDuplicate rows : {df_raw.duplicated().sum():,}")
print(f"\nBasic statistics (text length):")
df_raw['text_len'] = df_raw['text'].astype(str).apply(len)
print(df_raw['text_len'].describe())

In [ ]:
# ── ── 2b. Target / Label Distribution ───────────────────────────────────────
label_counts = df_raw['class'].value_counts()
print("\nClass Distribution:")
print(label_counts)
print(f"Imbalance ratio: {label_counts.iloc[0]/label_counts.iloc[1]:.2f}:1")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
label_counts.plot(kind='bar', ax=axes[0], color=['#4C6EF5', '#F03E3E'], edgecolor='white')
axes[0].set_title('Class Distribution', fontsize=14)
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Pie chart
axes[1].pie(label_counts, labels=label_counts.index, autopct='%1.1f%%',
            colors=['#4C6EF5', '#F03E3E'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportions', fontsize=14)

fig.tight_layout()
fig.savefig('outputs/plots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Class distribution plot saved")

In [ ]:
# ── ── 2c. Text Length Analysis ───────────────────────────────────────────────
df_raw['word_count'] = df_raw['text'].astype(str).apply(lambda x: len(x.split()))
df_raw['char_count'] = df_raw['text'].astype(str).apply(len)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (col, title) in enumerate([
    ('word_count', 'Word Count Distribution'),
    ('char_count', 'Character Count Distribution')
]):
    for j, (cls, color) in enumerate([('suicide', '#F03E3E'), ('non-suicide', '#4C6EF5')]):
        data = df_raw[df_raw['class'] == cls][col]
        axes[i][j].hist(data.clip(upper=data.quantile(0.99)), bins=50,
                         color=color, alpha=0.75, edgecolor='white')
        axes[i][j].set_title(f'{title} — {cls}', fontsize=12)
        axes[i][j].set_xlabel(col.replace('_', ' ').title())
        axes[i][j].set_ylabel('Frequency')
        axes[i][j].axvline(data.median(), color='black', linestyle='--',
                            label=f'Median: {data.median():.0f}')
        axes[i][j].legend()

fig.suptitle('Text Length Analysis by Class', fontsize=15, y=1.01)
fig.tight_layout()
fig.savefig('outputs/plots/text_length_analysis.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Text length analysis saved")

In [ ]:
# ── ── 2d. Boxplot — Word Count by Class ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
df_raw.boxplot(column='word_count', by='class', ax=ax,
               boxprops=dict(color='#4C6EF5'),
               medianprops=dict(color='#F03E3E', linewidth=2),
               whiskerprops=dict(color='#4C6EF5'),
               capprops=dict(color='#4C6EF5'),
               flierprops=dict(marker='o', color='gray', alpha=0.3, markersize=3))
ax.set_title('Word Count by Class (Boxplot)', fontsize=13)
ax.set_xlabel('Class')
ax.set_ylabel('Word Count')
plt.suptitle('')
fig.tight_layout()
fig.savefig('outputs/plots/boxplot_wordcount.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Boxplot saved")

In [ ]:
# ── ── 2e. Word Clouds ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (cls, color) in zip(axes, [('suicide', 'Reds'), ('non-suicide', 'Blues')]):
    texts = ' '.join(df_raw[df_raw['class'] == cls]['text'].astype(str).tolist())
    wc = WordCloud(
        width=700, height=400,
        background_color='white',
        colormap=color,
        max_words=150,
        collocations=False,
    ).generate(texts)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Word Cloud — {cls}', fontsize=14, pad=10)

fig.suptitle('Most Frequent Words by Class', fontsize=15)
fig.tight_layout()
fig.savefig('outputs/plots/wordclouds.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Word clouds saved")

In [ ]:
# ── ── 2f. Top N Words ────────────────────────────────────────────────────────
import re
from nltk.corpus import stopwords

STOP = set(stopwords.words('english'))

def top_words(series, n=20):
    words = []
    for text in series.astype(str):
        tokens = re.findall(r'\b[a-z]{3,}\b', text.lower())
        words.extend([t for t in tokens if t not in STOP])
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (cls, color) in zip(axes, [('suicide', '#F03E3E'), ('non-suicide', '#4C6EF5')]):
    words, counts = zip(*top_words(df_raw[df_raw['class'] == cls]['text']))
    ax.barh(words[::-1], counts[::-1], color=color, alpha=0.8)
    ax.set_title(f'Top 20 Words — {cls}', fontsize=13)
    ax.set_xlabel('Frequency')

fig.tight_layout()
fig.savefig('outputs/plots/top_words.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Top words plot saved")

In [ ]:
# ── ── 2g. N-gram Analysis ────────────────────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer

def top_ngrams(series, n=2, top=15):
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english', max_features=5000)
    X = vec.fit_transform(series.astype(str))
    counts = X.sum(axis=0).A1
    vocab  = vec.get_feature_names_out()
    return sorted(zip(vocab, counts), key=lambda x: -x[1])[:top]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row, (cls, color) in enumerate([('suicide', '#F03E3E'), ('non-suicide', '#4C6EF5')]):
    data = df_raw[df_raw['class'] == cls]['text']
    for col, n in enumerate([2, 3]):
        ngrams_top = top_ngrams(data, n=n, top=15)
        phrases, counts = zip(*ngrams_top)
        axes[row][col].barh(phrases[::-1], counts[::-1], color=color, alpha=0.8)
        axes[row][col].set_title(
            f'Top {"Bigrams" if n==2 else "Trigrams"} — {cls}', fontsize=12
        )

fig.suptitle('N-gram Analysis by Class', fontsize=15, y=1.01)
fig.tight_layout()
fig.savefig('outputs/plots/ngrams.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ N-gram plots saved")

## 3. Data Quality Checks

In [ ]:
# ── ── 3. Data Quality ────────────────────────────────────────────────────────
print("=" * 55)
print("DATA QUALITY CHECKS")
print("=" * 55)

# Skewness
for col in ['word_count', 'char_count']:
    skew = df_raw[col].skew()
    print(f"\nSkewness of {col}: {skew:.3f} "
          f"({'right-skewed' if skew > 0 else 'left-skewed'})")

# Outlier detection using IQR
print("\nOutlier Detection (IQR method):")
for col in ['word_count', 'char_count']:
    Q1 = df_raw[col].quantile(0.25)
    Q3 = df_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df_raw[(df_raw[col] < Q1 - 1.5*IQR) | (df_raw[col] > Q3 + 1.5*IQR)]
    print(f"  {col}: {len(outliers):,} outliers ({len(outliers)/len(df_raw)*100:.1f}%)")

# Very short / empty posts
short = df_raw[df_raw['word_count'] < 3]
print(f"\nPosts with < 3 words: {len(short):,}")

## 4. Preprocessing

In [ ]:
# ── ── 4. Preprocessing ───────────────────────────────────────────────────────
print("Applying preprocessing pipeline …")
df = preprocess_dataframe(df_raw, text_col='text', label_col='class')
print(f"\nShape after preprocessing: {df.shape}")
print(df.head(3))

# Save cleaned dataset for Gradio app
df.to_csv('data/suicide_detection.csv', index=False)
print("✓ Cleaned dataset saved → data/suicide_detection.csv")

## 5. Feature Engineering

In [ ]:
# ── ── 5. Feature Engineering ─────────────────────────────────────────────────
tfidf = build_tfidf_vectorizer(max_features=50_000, ngram_range=(1, 2))

X = build_feature_matrix(
    clean_texts=df['clean_text'],
    raw_texts=df['text'],
    tfidf=tfidf,
    fit=True,
)
y = df['label'].values

print(f"Feature matrix shape : {X.shape}")
print(f"Label distribution   : {dict(zip(*np.unique(y, return_counts=True)))}")

save_vectorizer(tfidf, 'models/tfidf_vectorizer.joblib')

# Handcrafted feature distributions
hand_feat = extract_handcrafted_features(df['text'])
hand_df = pd.DataFrame(hand_feat, columns=HANDCRAFTED_FEATURE_NAMES)
hand_df['label'] = y

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()
for i, col in enumerate(HANDCRAFTED_FEATURE_NAMES):
    for cls, color in [(0, '#4C6EF5'), (1, '#F03E3E')]:
        axes[i].hist(hand_df[hand_df['label']==cls][col],
                     bins=30, alpha=0.6, color=color,
                     label='Non-Suicide' if cls==0 else 'Suicide')
    axes[i].set_title(col, fontsize=10)
    axes[i].legend(fontsize=8)

fig.suptitle('Handcrafted Feature Distributions by Class', fontsize=15)
fig.tight_layout()
fig.savefig('outputs/plots/handcrafted_features.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Handcrafted feature distributions saved")

# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 9))
corr = hand_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Handcrafted Feature Correlation Matrix', fontsize=13)
fig.tight_layout()
fig.savefig('outputs/plots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✓ Correlation heatmap saved")

## 6. Train / Test Split

In [ ]:
# ── ── 6. Train / Test Split ──────────────────────────────────────────────────
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.20)

print(f"Training set  : {X_train.shape[0]:,} samples")
print(f"Test set      : {X_test.shape[0]:,} samples")
print(f"Train positives: {y_train.sum():,}  ({y_train.mean()*100:.1f}%)")
print(f"Test  positives: {y_test.sum():,}  ({y_test.mean()*100:.1f}%)")

## 7. Model Training

In [ ]:
# ── ── 7. Model Training ───────────────────────────────────────────────────────
# Set tune=True to run RandomizedSearchCV (slower but finds better hyper-params)
trained_models = train_all_models(X_train, y_train, tune=False)
save_models(trained_models, directory='models/')
print("\n✓ All models trained and saved")

## 8. Evaluation

In [ ]:
# ── ── 8. Evaluation ──────────────────────────────────────────────────────────
comparison_df = full_evaluation(trained_models, X_test, y_test)

print("\n" + "="*55)
print("FINAL COMPARISON TABLE (sorted by F1-Score)")
print("="*55)
print(comparison_df.to_string(float_format="{:.4f}".format))

best_model_name = comparison_df.index[0]
best_model = trained_models[best_model_name]
print(f"\n🏆 Best model: {best_model_name}")

## 9. LLM Explanation (Groq)

In [ ]:
# ── ── 9. LLM Integration ─────────────────────────────────────────────────────
# Set your Groq API key:
GROQ_API_KEY = GROQ_API_KEY = "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"  # or paste key directly

sample_texts = [
    "I've been feeling so hopeless lately. I don't see any reason to keep going on.",
    "Just had a great workout! Feeling energized and ready to tackle the day.",
]

for text in sample_texts:
    clean = preprocess_text(text)

    # Build feature for single sample
    from scipy.sparse import csr_matrix
    clean_s = pd.Series([clean])
    raw_s   = pd.Series([text])
    X_sample = build_feature_matrix(clean_s, raw_s, tfidf, fit=False)

    pred = best_model.predict(X_sample)[0]
    if hasattr(best_model, 'predict_proba'):
        conf = best_model.predict_proba(X_sample)[0][pred] * 100
    else:
        conf = 85.0  # fallback

    label_str = "Suicide Risk" if pred == 1 else "Non-Suicide"

    print(f"\n{'─'*60}")
    print(f"Text      : {text[:80]}…")
    print(f"Prediction: {label_str}  ({conf:.1f}%)")

    if GROQ_API_KEY:
        print("\nGroq Explanation:")
        explanation = explain_prediction(text, label_str, conf, api_key=GROQ_API_KEY)
        print(explanation)
    else:
        print("\n⚠️  Set GROQ_API_KEY to see the LLM explanation.")

## 10. Launch Gradio App

In [ ]:
# ── ── 10. Gradio App ─────────────────────────────────────────────────────────
print("\nLaunching Gradio app …")
print("A public share link will be displayed below.\n")

# Uncomment the line below to launch:
# %run app_gradio.py

print("To launch the Gradio app, run: !python app_gradio.py")
print("\n✓ Full pipeline complete!")